In [3]:
# ============================================================
# TELEQnA QLoRA FINE-TUNING
# Qwen2.5-1.5B-Instruct
# ============================================================

import json
import torch

from pathlib import Path
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import LoraConfig

from trl import (
    SFTConfig,
    SFTTrainer,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

DATA_FILE = Path(r"TeleQnA.json")

OUTPUT_DIR = "./teleqna-qwen-1.5b-qlora"

SEED = 42


# ============================================================
# 2. CHECK GPU
# ============================================================

print("=" * 70)
print("SYSTEM")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected. "
        "QLoRA training is not configured for CPU."
    )

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2
    ),
    "GB"
)


# ============================================================
# 3. LOAD TELEQnA
# ============================================================

print("\n" + "=" * 70)
print("LOADING TELEQnA")
print("=" * 70)

with open(
    DATA_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

print("Total questions:", len(data))


# ============================================================
# 4. CONVERT TELEQnA → CHAT FORMAT
# ============================================================

records = []

for item in data.values():

    question = item["question"]

    option_keys = [k for k in item.keys() if k.startswith("option")]
    options = "\n".join(f"{i+1}.  {item[k]}" for i, k in enumerate(option_keys))

    user_content = (
        f"Question:\n"
        f"{question}\n\n"
        f"Options:\n"
        f"{options}"
    )

    assistant_content = item["answer"]

    records.append({
        "messages": [
            {
                "role": "user",
                "content": user_content
            },
            {
                "role": "assistant",
                "content": assistant_content
            }
        ]
    })


print("Converted examples:", len(records))


# ============================================================
# 5. CREATE DATASET
# ============================================================

dataset = Dataset.from_list(records)

print("\nExample:")
print(dataset[0])


# ============================================================
# 6. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

dataset = dataset.shuffle(seed=SEED)

train_test = dataset.train_test_split(
    test_size=0.20,
    seed=SEED
)

train_dataset = train_test["train"]

test_dataset = train_test["test"]

validation_test = test_dataset.train_test_split(
    test_size=0.50,
    seed=SEED
)

validation_dataset = validation_test["train"]

test_dataset = validation_test["test"]


print("\n" + "=" * 70)
print("DATA SPLIT")
print("=" * 70)

print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))


# ============================================================
# 7. TOKENIZER
# ============================================================

print("\n" + "=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ------------------------------------------------------------
# Inject the {% generation %} keyword into the chat template
# so TRL can mask non-assistant tokens when
# assistant_only_loss=True.
# ------------------------------------------------------------

template = tokenizer.chat_template

marker = (
    '{%- if (message.role == "user") or '
    '(message.role == "system" and not loop.first) or '
    '(message.role == "assistant" and not message.tool_calls) %}\n'
    "        {{- '<|im_start|>' + message.role + '\\n' + "
    "message.content + '<|im_end|>' + '\\n' }}"
)

replacement = (
    '{%- if (message.role == "user") or '
    '(message.role == "system" and not loop.first) %}\n'
    "        {{- '<|im_start|>' + message.role + '\\n' + "
    "message.content + '<|im_end|>' + '\\n' }}\n"
    '    {%- elif (message.role == "assistant" and '
    'not message.tool_calls) %}\n'
    "        {{- '<|im_start|>assistant\\n' }}"
    "{% generation %}{{ message.content + '<|im_end|>' + '\\n' }}"
    "{% endgeneration %}"
)

assert marker in template, 'Template marker not found'

tokenizer.chat_template = template.replace(marker, replacement)


# ============================================================
# 8. 4-BIT QUANTIZATION
# ============================================================

print("\n" + "=" * 70)
print("4-BIT QUANTIZATION")
print("=" * 70)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True,

    bnb_4bit_compute_dtype=torch.bfloat16,
)


# ============================================================
# 9. LOAD BASE MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING MODEL")
print("=" * 70)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,

    quantization_config=bnb_config,

    # Pin the whole model to the GPU. With only 4 GB of VRAM,
    # device_map="auto" can estimate there is not enough room and
    # offload modules to CPU/disk, which the 4-bit quantizer rejects.

    device_map={"": 0},

    torch_dtype=torch.torch.bfloat16,
)

model.config.use_cache = False


# ============================================================
# 10. LoRA CONFIGURATION
# ============================================================

print("\n" + "=" * 70)
print("LoRA CONFIGURATION")
print("=" * 70)

peft_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)


# ============================================================
# 11. TRAINING CONFIGURATION
# ============================================================

training_args = SFTConfig(

    output_dir=OUTPUT_DIR,

    # ------------------------------
    # Training
    # ------------------------------

    num_train_epochs=1,

    learning_rate=2e-4,

    # ------------------------------
    # Memory
    # ------------------------------

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    gradient_checkpointing=False,

    # ------------------------------
    # Precision
    # ------------------------------

    fp16=False,

    bf16=True,

    # ------------------------------
    # Sequence length
    # ------------------------------

    max_length=512,

    # ------------------------------
    # Logging
    # ------------------------------

    logging_steps=10,

    logging_first_step=True,

    # ------------------------------
    # Evaluation
    # ------------------------------

    eval_strategy="steps",

    eval_steps=100,

    # ------------------------------
    # Saving
    # ------------------------------

    save_strategy="steps",

    save_steps=100,

    save_total_limit=2,

    # ------------------------------
    # Misc
    # ------------------------------

    report_to="none",

    seed=SEED,

    # We are training a conversational dataset.
    assistant_only_loss=True,
)


# ============================================================
# 12. CREATE TRAINER
# ============================================================

print("\n" + "=" * 70)
print("CREATING TRAINER")
print("=" * 70)

trainer = SFTTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=validation_dataset,

    processing_class=tokenizer,

    peft_config=peft_config,
)


# ============================================================
# 13. SHOW TRAINABLE PARAMETERS
# ============================================================

print("\n" + "=" * 70)
print("TRAINABLE PARAMETERS")
print("=" * 70)

trainer.model.print_trainable_parameters()


# ============================================================
# 14. TRAIN
# ============================================================

print("\n" + "=" * 70)
print("STARTING QLoRA TRAINING")
print("=" * 70)

trainer.train()


# ============================================================
# 15. SAVE LoRA ADAPTER
# ============================================================

print("\n" + "=" * 70)
print("SAVING LoRA ADAPTER")
print("=" * 70)

trainer.save_model(OUTPUT_DIR)

tokenizer.save_pretrained(OUTPUT_DIR)

print(
    "\nAdapter saved to:",
    OUTPUT_DIR
)


# ============================================================
# 16. FINAL EVALUATION LOSS
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

metrics = trainer.evaluate()

print(metrics)

print("\nTraining finished.")

SYSTEM
PyTorch: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA L40S
VRAM: 44.39 GB

LOADING TELEQnA
Total questions: 10000
Converted examples: 10000

Example:
{'messages': [{'role': 'user', 'content': 'Question:\nWhat is the purpose of the Nmfaf_3daDataManagement_Deconfigure service operation? [3GPP Release 18]\n\nOptions:\n1.  To configure the MFAF to map data or analytics received by the MFAF to out-bound notification endpoints\n2.  To configure the MFAF to stop mapping data or analytics received by the MFAF to out-bound notification endpoints\n3.  To supply data or analytics from the MFAF to notification endpoints\n4.  To fetch data or analytics from the MFAF based on fetch instructions'}, {'role': 'assistant', 'content': 'option 2: To configure the MFAF to stop mapping data or analytics received by the MFAF to out-bound notification endpoints'}]}

DATA SPLIT
Train: 8000
Validation: 1000
Test: 1000

LOADING TOKENIZER


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



4-BIT QUANTIZATION

LOADING MODEL


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


LoRA CONFIGURATION

CREATING TRAINER


Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

... (9 outputs left out)

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.050585,0.065807,0.049236,190784.000000,0.975411
200,0.058986,0.058991,0.058061,382013.000000,0.976164
300,0.053721,0.056870,0.055029,574059.000000,0.977160
400,0.058178,0.056131,0.058873,763566.000000,0.977231
500,0.069312,0.055505,0.059846,956170.000000,0.977468



SAVING LoRA ADAPTER

Adapter saved to: ./teleqna-qwen-1.5b-qlora

VALIDATION


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
0.069312,0.055505,500,0.059846,956170.000000,0.977468


{'eval_loss': 0.05550482124090195, 'eval_entropy': 0.059845883086323735, 'eval_num_tokens': 956170.0, 'eval_mean_token_accuracy': 0.9774681072235107}

Training finished.


In [4]:
import torch

# After trainer creation
print("Model device:", next(trainer.model.parameters()).device)
print("Model dtype:", next(trainer.model.parameters()).dtype)

# During/after training
print("CUDA memory allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("CUDA memory reserved:", torch.cuda.memory_reserved() / 1024**3, "GB")
print("Max memory allocated:", torch.cuda.max_memory_allocated() / 1024**3, "GB")

Model device: cuda:0
Model dtype: torch.bfloat16
CUDA memory allocated: 1.114539623260498 GB
CUDA memory reserved: 19.080078125 GB
Max memory allocated: 11.218731880187988 GB


In [ ]:
pip install "transformers==4.52.3" "trl==0.17.0" "accelerate>=0.34.0" "bitsandbytes==0.46.1"

In [ ]:
from transformers.utils import is_bitsandbytes_available
print("bitsandbytes available:", is_bitsandbytes_available())

In [ ]:
pip install "trl>=0.21"